In [ ]:
import gym_super_mario_bros
from nes_py.wrappers import JoypadSpace
from gym_super_mario_bros.actions import RIGHT_ONLY

In [ ]:
# Escopo de ações:
env = gym_super_mario_bros.make('SuperMarioBros-1-1-v0')
env = JoypadSpace(env, RIGHT_ONLY)
print(env.action_space)

In [ ]:
# Visualizando estados:
import time

env = gym_super_mario_bros.make('SuperMarioBros-1-1-v0')
env = JoypadSpace(env, RIGHT_ONLY)
obs = env.reset()

for i in range(500):
    env.render()
    time.sleep(0.02)
    acao = 1
    obs, recompensa, terminou, info = env.step(acao)
    print(recompensa)
    if terminou:
        print("Episódio finalizado depois de {} steps".format(i+1))
        break
env.close()

In [ ]:
import gym_super_mario_bros
from nes_py.wrappers import JoypadSpace
from gym_super_mario_bros.actions import RIGHT_ONLY
import time
from OpenAIWrappers import nosso_wrapper
from stable_baselines3.common.vec_env import SubprocVecEnv
from stable_baselines3 import PPO
from stable_baselines3.common.callbacks import CheckpointCallback

def cria_env(nome_env):
    def _init():
        env = gym_super_mario_bros.make(nome_env)
        env = JoypadSpace(env, RIGHT_ONLY)
        env = nosso_wrapper(env)
        return env
    return _init


threads_cpu = 16
env = SubprocVecEnv([cria_env(nome_env='SuperMarioBros-1-1-v0') for i in range(threads_cpu)])

checkpoint_callback = CheckpointCallback(save_freq=6250, save_path='C:/Users/natan/Documents/modelos_salvos/', name_prefix='n1024b64l3')

modelo = PPO(policy='CnnPolicy', env=env, gamma=0.99, n_steps=1024, batch_size=64, learning_rate=0.0003, vf_coef=0.5, ent_coef=0.01, verbose=1)

inicio = time.time()
modelo.learn(total_timesteps=10000000, log_interval=10, callback=checkpoint_callback)

fim = time.time()
print('Tempo total do treinamento em horas:', (fim-inicio)/3600)